# Fusang vs Mash: MinHash Jaccard vs k-mer Cosine Distance
## Supplementary Table S14 — 30-seed indel-rich benchmark

This notebook reproduces the Mash vs spaced k-mer benchmark (Supplementary Table S14).
It compares **MinHash Jaccard distance** (Mash, k=21, NJ) against **spaced k-mer frequency vector cosine distance**
(Fusang, k=5,gap2, NJ), both evaluated against TRUE coalescent trees under clean and indel-rich (indel=0.02) conditions.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'fusang'))

import numpy as np
import statistics as st
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import csv

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

In [ ]:
# Load IMMI (Fusang) 30-seed data from JSON
with open('../data/mash_vs_immi_results.json') as f:
    data = json.load(f)

immi_clean_nrfs = data['immi_clean_nrfs']
immi_indel_nrfs = data['immi_indel_nrfs']

# Load Mash 30-seed data from CSV
mash_clean_nrfs = []
mash_indel_nrfs = []
with open('../data/mash_benchmark_30seeds.csv') as f:
    for row in csv.DictReader(f):
        mash_clean_nrfs.append(float(row['clean_nrf']))
        mash_indel_nrfs.append(float(row['indel_nrf']))

print(f'IMMI clean:  {st.mean(immi_clean_nrfs):.3f} ± {st.stdev(immi_clean_nrfs):.3f} (n={len(immi_clean_nrfs)})')
print(f'IMMI indel:  {st.mean(immi_indel_nrfs):.3f} ± {st.stdev(immi_indel_nrfs):.3f} (n={len(immi_indel_nrfs)})')
print(f'Mash clean:  {st.mean(mash_clean_nrfs):.3f} ± {st.stdev(mash_clean_nrfs):.3f} (n={len(mash_clean_nrfs)})')
print(f'Mash indel:  {st.mean(mash_indel_nrfs):.3f} ± {st.stdev(mash_indel_nrfs):.3f} (n={len(mash_indel_nrfs)})')

In [ ]:
# Statistical comparison: Fusang vs Mash
print('='*60)
print('CLEAN DATA (n=200, no indel)')
print('='*60)
w_clean, p_clean = stats.wilcoxon(immi_clean_nrfs, mash_clean_nrfs)
d_clean = (st.mean(immi_clean_nrfs) - st.mean(mash_clean_nrfs)) / np.sqrt(
    (st.stdev(immi_clean_nrfs)**2 + st.stdev(mash_clean_nrfs)**2) / 2)
print(f'Fusang: {st.mean(immi_clean_nrfs):.3f} ± {st.stdev(immi_clean_nrfs):.3f}')
print(f'Mash:   {st.mean(mash_clean_nrfs):.3f} ± {st.stdev(mash_clean_nrfs):.3f}')
print(f'Ratio (Mash/Fusang): {st.mean(mash_clean_nrfs)/st.mean(immi_clean_nrfs):.2f}x')
print(f'Wilcoxon p = {p_clean:.4f}, Cohen d = {d_clean:.2f}')

print()
print('='*60)
print('INDEL-RICH DATA (n=200, indel=0.02)')
print('='*60)
w_indel, p_indel = stats.wilcoxon(immi_indel_nrfs, mash_indel_nrfs)
d_indel = (st.mean(immi_indel_nrfs) - st.mean(mash_indel_nrfs)) / np.sqrt(
    (st.stdev(immi_indel_nrfs)**2 + st.stdev(mash_indel_nrfs)**2) / 2)
print(f'Fusang: {st.mean(immi_indel_nrfs):.3f} ± {st.stdev(immi_indel_nrfs):.3f}')
print(f'Mash:   {st.mean(mash_indel_nrfs):.3f} ± {st.stdev(mash_indel_nrfs):.3f}')
print(f'Ratio (Mash/Fusang): {st.mean(mash_indel_nrfs)/st.mean(immi_indel_nrfs):.2f}x')
print(f'Wilcoxon p = {p_indel:.4f}, Cohen d = {d_indel:.2f}')

print()
print('CONCLUSION: Spaced k-mer cosine is', end=' ')
print(f'{st.mean(mash_clean_nrfs)/st.mean(immi_clean_nrfs):.1f}x more robust than MinHash Jaccard on clean data,', end=' ')
print(f'{st.mean(mash_indel_nrfs)/st.mean(immi_indel_nrfs):.1f}x on indel-rich data.')

In [ ]:
# Figure: Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clean data
ax = axes[0]
data_clean = [immi_clean_nrfs, mash_clean_nrfs]
vp = ax.violinplot(data_clean, positions=[1, 2], showmeans=True, showmedians=True)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Fusang\n(k-mer cosine)', 'Mash\n(MinHash Jaccard)'])
ax.set_ylabel('Normalized RF Distance')
ax.set_title(f'Clean Data (n=200, 30 seeds)\nWilcoxon p={p_clean:.4f}')
ax.set_ylim(0, 0.7)

# Indel-rich data
ax = axes[1]
data_indel = [immi_indel_nrfs, mash_indel_nrfs]
vp = ax.violinplot(data_indel, positions=[1, 2], showmeans=True, showmedians=True)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Fusang\n(k-mer cosine)', 'Mash\n(MinHash Jaccard)'])
ax.set_ylabel('Normalized RF Distance')
ax.set_title(f'Indel-Rich (n=200, indel=0.02, 30 seeds)\nWilcoxon p={p_indel:.4f}')
ax.set_ylim(0, 1.0)

plt.suptitle('Spaced k-mer Cosine vs MinHash Jaccard Distance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/mash_vs_fusang_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-seed delta: Fusang - Mash (negative = Fusang better)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

diff_clean = [immi_clean_nrfs[i] - mash_clean_nrfs[i] for i in range(30)]
diff_indel = [immi_indel_nrfs[i] - mash_indel_nrfs[i] for i in range(30)]

colors_clean = ['#2ecc71' if d < 0 else '#e74c3c' for d in diff_clean]
ax1.bar(range(30), diff_clean, color=colors_clean, alpha=0.7)
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.axhline(y=st.mean(diff_clean), color='blue', linestyle='--', label=f'Mean={st.mean(diff_clean):.4f}')
ax1.set_xlabel('Seed index')
ax1.set_ylabel('nRF difference (Fusang - Mash)')
ax1.set_title(f'Clean\nFusang better in {sum(1 for d in diff_clean if d<0)}/30 seeds')
ax1.legend()

colors_indel = ['#2ecc71' if d < 0 else '#e74c3c' for d in diff_indel]
ax2.bar(range(30), diff_indel, color=colors_indel, alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.axhline(y=st.mean(diff_indel), color='blue', linestyle='--', label=f'Mean={st.mean(diff_indel):.4f}')
ax2.set_xlabel('Seed index')
ax2.set_ylabel('nRF difference (Fusang - Mash)')
ax2.set_title(f'Indel-rich\nFusang better in {sum(1 for d in diff_indel if d<0)}/30 seeds')
ax2.legend()

plt.suptitle('Per-Seed Comparison: Fusang(k-mer cosine) vs Mash(MinHash Jaccard)', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/mash_vs_fusang_per_seed.png', dpi=150, bbox_inches='tight')
plt.show()

## Reproducing the Mash Benchmark

The Mash 30-seed benchmark was computed using:

```bash
# Step 1: Generate Mash sketches (one per sequence)
for seed in {1..30}; do
    mash sketch -i $seed.fasta -o $seed.msh -k 21 -s 1000
done

# Step 2: Compute pairwise distances
mash triangle *.msh > mash_distances.tsv

# Step 3: Build NJ tree and compute nRF
python scripts/compute_mash_vs_immi.py --mash-dist mash_distances.tsv --true-trees true_trees/
```

**Requirements**: [Mash v2.3+](https://github.com/marbl/Mash) (external tool, not in requirements.txt)